# Student fine-tuning: Qwen3-ASR Khmer LoRA
Train the published Khmer checkpoint with a **student-owned LoRA adapter** on exactly 531 manifested FLEURS training clips. Use exactly 124 validation clips for selection. The script does not load the held-out test split. A validation result is not a final test claim. CER and ICU Khmer WER must each be below 20% to exceed 80% correctness under these measures.


In [ ]:
!nvidia-smi
!pip install -q qwen-asr datasets torchcodec pyicu-wheels==2.15.2 jiwer soundfile accelerate peft
!pip uninstall -y torchao
from pathlib import Path
import subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
repo = Path('/content/khmer_asr')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Seypa-47/khmer_asr.git', str(repo)], check=True)
official = Path('/content/Qwen3-ASR')
if not official.exists():
    subprocess.run(['git', 'clone', 'https://github.com/QwenLM/Qwen3-ASR.git', str(official)], check=True)
subprocess.run(['git', '-C', str(official), 'checkout', '7c6daf77a2421100f5fb066495372c00129d39ff'], check=True)
%cd /content/khmer_asr


Mount Drive only after reviewing the code above. Save the small adapter checkpoints there so a Colab disconnect does not erase them. If Drive mounting fails, use a Colab runtime path temporarily and download the checkpoint folder before disconnecting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# One controlled epoch; checkpoints and validation predictions stay outside Git.
!python src/train_qwen_lora.py --output-dir /content/drive/MyDrive/khmer_asr_final_runs/qwen_lora_v1 --epochs 1 --lr 1e-4 --grad-acc 4


Compare the full validation CER/WER with the published checkpoint's 14.29% CER and 32.07% WER on the same 124 clips. Only after choosing an adapter should the separate held-out test be evaluated once. Do not present the publisher's pre-trained model as a student-trained approach.
